# `run_externalities.ipynb` — dataset → alignment externalities

Thin driver over the `hiddenstructure` package. Pick a dataset + mechanisms in
the config cell below and run all cells to get the Γ externality matrices and
heatmaps. All the math lives in `src/hiddenstructure/` (imported, not inlined),
so this notebook stays short and every dataset/mechanism shares one code path.

**Pipeline:** `load_dataset → fit_model → mechanism.deploy/loo → Γ_{n,m} → heatmaps`


## Setup — put the package on the path

Data is read from `../distortion` by default (set `HS_DATA_ROOT` to override).

In [ ]:
import sys, os
sys.path.insert(0, "src")
# Optional: point at the data copy explicitly.
# os.environ["HS_DATA_ROOT"] = "/Users/michellesi/Desktop/harvard/distortion"

%config InlineBackend.figure_formats = ['svg']
import numpy as np
import matplotlib.pyplot as plt
try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
except ImportError:
    pass
plt.rcParams["figure.dpi"] = 110

import hiddenstructure as hs
from hiddenstructure import pipeline, plotting
from hiddenstructure.mechanisms import (
    Utilitarian, RLHF, Strategyproof, BoundedHarm, PublicSpirit)
print("data root:", hs.config.data_root())

## Config

- `DATASET` — one of `"412 Food Rescue"`, `"Kidney Allocation"`, `"Moral Machine"`, `"Community Alignment"`.
- `MECHANISMS` — any mix of `Utilitarian()`, `RLHF()`, `Strategyproof()`,
  `BoundedHarm(eps=...)`, `PublicSpirit(gamma=...)`.
- `STRATIFIED_N` — cap the agent count (stratified by `STRATIFY_BY`) so the leave-one-out refits stay tractable; `None` uses all agents.

In [ ]:
DATASET      = "Community Alignment"
MECHANISMS   = [Utilitarian(), RLHF(), Strategyproof()]
# welfare-constraint mechanisms (pipeline #2) plug in the same way:
# MECHANISMS = [Utilitarian(), BoundedHarm(eps=0.05), PublicSpirit(gamma=0.5)]
STRATIFIED_N = 100          # None to use all agents
STRATIFY_BY  = "country"
BETA         = 1.0

## Run the pipeline

One call: load → BTL fits → per-mechanism deployment ψ + leave-one-out ψ⁽⁻ⁿ⁾ → Γ.

In [ ]:
run = pipeline.alignment_externalities(
    DATASET, mechanisms=MECHANISMS,
    stratified_n=STRATIFIED_N, stratify_by=STRATIFY_BY,
    beta=BETA, use_cache=True, verbose=True)

model = run.model
print(f"\n{DATASET}:  N={model.N} agents,  k={model.k} features")
print("group axes:", {k: len(np.unique(v)) for k, v in model.group_axes.items()})
for name, res in run.results.items():
    print(res.summary())

## §1 Per-agent Γ heatmaps

Row $n$ = the externality agent $n$'s participation imposes; column $m$ = what
agent $m$ receives. $\Gamma_{n,m} = \theta_m^\top(\psi(\theta_c) - \psi(\theta_c^{(-n)}))$.
Agents are sorted by their primary group label; each panel clips at its 99th percentile.

In [ ]:
fig = plotting.plot_per_agent(run.results, model)
plt.show()

## §2 Group-averaged Γ heatmaps

Every grouping axis (demographics, or k-means clusters when none exist) × every mechanism. Off-diagonal blocks are cross-group externalities; the diagonal excludes self-pairs.

In [ ]:
fig = plotting.plot_group_averaged(run.results, model)
plt.show()

## §3 Headlines — who imposes / receives externalities

Row sums = total externality *imposed* by each agent; column sums = total *received*.

In [ ]:
for name, res in run.results.items():
    G = res.Gamma
    imposed  = G.sum(axis=1)   # row n: n's effect on everyone
    received = G.sum(axis=0)   # col m: everyone's effect on m
    top_imp  = model.agent_ids[np.argsort(imposed)[::-1][:5]]
    top_rec  = model.agent_ids[np.argsort(received)[::-1][:5]]
    print(f"[{name}]")
    print(f"  biggest imposers : {list(top_imp)}")
    print(f"  biggest receivers: {list(top_rec)}\n")

---
### Pipeline #2 mechanisms — bounded harm & public spirit

Both welfare-constraint mechanisms are implemented and plug into the exact same
interface — add `BoundedHarm(eps=0.05)` and/or `PublicSpirit(gamma=0.5)` to
`MECHANISMS` above to get their Γ heatmaps alongside the core three.

- **Bounded harm** — maximise utilitarian welfare s.t. a per-agent harm floor −ε·h(θ_n).
- **Public spirit** (Flanigan et al. 2023) — each agent tolerates personal harm in
  proportion to social benefit; `gamma=0` is individual rationality (no agent worse
  than the reference impact ψ₀), `gamma=1` collapses to utilitarian.